# v0.2 — CIFAR-10 CNN inference on AUP-ZU3

End-to-end hardware inference of the v0.2 student model on the
AUP-ZU3 board (Zynq UltraScale+ XCZU3EG).

Pipeline that produced the bitstream loaded by this notebook:
1. `01.training` — Keras + QKeras distilled student, ~14 K params, 8-bit weights, 61.41% software accuracy.
2. `02.hls4ml` — hls4ml conversion to Vitis HLS C++, `ap_fixed<16,6>` precision, bit-accurate 60.97%.
3. `03.hls` — AXI-Stream + AXI-Lite wrapper exposing `s_axis_in`, `s_axis_out`, `s_axi_CTRL`.
4. `04.hw` — Vivado block design (PS + DMA + interconnect + our IP), bitstream + XSA.
5. `05.pynq` — **this notebook**.

Each cell is intentionally small so we can validate the integration step by step.

## 1. Imports and environment

Sanity check before touching the hardware. If `pynq` is not importable
or `numpy` is missing, nothing else will work.

In [ ]:
import time
import numpy as np
from pynq import Overlay, allocate
from utils import (
    float_to_axis_word,
    axis_word_to_float,
    image_to_axis_buffer,
    axis_buffer_to_floats,
    confusion_matrix_np,
    plot_confusion_matrix,
    CIFAR10_CLASSES,
    FX_FRAC_BITS,
)

print(f'numpy version: {np.__version__}')
print(f'Fixed-point: 16 bits total, {FX_FRAC_BITS} fractional bits (ap_fixed<16,6>)')

## 2. Load the FPGA overlay

This is the **first critical step**. If the bitstream is malformed,
if the metadata inside the XSA is incompatible with the running PYNQ
image, or if the board fails to program the PL, this is where it will
explode.

Expected: the cell returns silently (the kernel does not die).

In [ ]:
ol = Overlay('bd_wrapper_cifar10.xsa')
print('Overlay loaded.')

## 3. Discover IP cores in the overlay

Should print three names:
- `axi_dma_0`
- `myproject_cifar10_accel_0`
- `zynq_ultra_ps_e_0`

If the accelerator name differs (e.g. it does not show as
`myproject_cifar10_accel_0`), the HLS wrapper might have been packaged
with a different VLNV. Adjust the variable `IP_NAME` below.

In [ ]:
list(ol.ip_dict.keys())

## 4. Create handles for DMA and the custom IP

AXI DMA channels:
- `sendchannel` — PS → PL (image input to the accelerator).
- `recvchannel` — PL → PS (10 class scores from the accelerator).

AXI-Lite control register at offset 0x00: writing `1` triggers the IP.

In [ ]:
IP_NAME = 'myproject_cifar10_accel_0'
IP_CTRL_REG = 0x00

dma = ol.axi_dma_0
dma_send = dma.sendchannel
dma_recv = dma.recvchannel

ip = getattr(ol, IP_NAME)
print(f'DMA handle: {dma}')
print(f'IP handle:  {ip}')

## 5. Allocate DMA-coherent buffers

- Input: 3072 AXI beats of 32 bits each — one for every (pixel, channel) value of a 32x32x3 CIFAR-10 image. Each beat carries a 16-bit `fixed<16,6>` in its low half.
- Output: 10 AXI beats, one per class score (also `fixed<16,6>`, low half of the 32-bit word).

`allocate` returns a buffer backed by physically contiguous memory accessible to the AXI DMA.

In [ ]:
N_IN  = 3072  # 32 * 32 * 3
N_OUT = 10

in_buffer  = allocate(shape=(N_IN,),  dtype=np.uint32)
out_buffer = allocate(shape=(N_OUT,), dtype=np.uint32)

print(f'in_buffer:  shape={in_buffer.shape},  dtype={in_buffer.dtype},  phys_addr=0x{in_buffer.physical_address:08x}')
print(f'out_buffer: shape={out_buffer.shape}, dtype={out_buffer.dtype}, phys_addr=0x{out_buffer.physical_address:08x}')

## 6. Load the CIFAR-10 test dataset

We use the CSV exported from the host (see the appendix at the bottom of this notebook for the export recipe).

Format:
- 3072 columns of float pixels in [0, 1] (already normalised by /255), in (H, W, C) order with C running fastest.
- Last column: integer label 0-9.

Note: the PYNQ 3.1.1 image does not ship with TensorFlow, so the CSV must be pre-generated on the host.

In [ ]:
import os
import pandas as pd

CSV_PATH = 'test_dataset/cifar10_test.csv'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f'{CSV_PATH} not found. Generate it on the host with the recipe '
        f'at the bottom of this notebook and SCP it to the board.'
    )

print(f'Loading {CSV_PATH} ...')
data = pd.read_csv(CSV_PATH).values
pixels = data[:, :-1].astype(np.float32)
labels = data[:, -1].astype(int)

N = pixels.shape[0]
print(f'Loaded {N} samples, pixel range [{pixels.min():.3f}, {pixels.max():.3f}]')

## 7. Visual sanity check on a single image

Show the first test image and its ground-truth label. Confirms that the (H, W, C) flattening is right and that the pixel range looks like a real image (not garbage).

If the image looks scrambled, the channel order or the reshape is wrong, and the model will not learn anything from the hardware either.

In [ ]:
import matplotlib.pyplot as plt

idx = 0
img = pixels[idx].reshape(32, 32, 3)
label = labels[idx]

plt.figure(figsize=(2, 2))
plt.imshow(img)
plt.title(f'idx={idx}, label={label} ({CIFAR10_CLASSES[label]})')
plt.axis('off')
plt.show()

## 8. Run inference on a single image

Step-by-step version of `inference_image`. Useful for debugging:
if any of the DMA `transfer`/`wait` calls hangs or the IP refuses to
raise its `ap_done` flag, this is where you will see it.

Expected behaviour: prints 10 floats, the argmax of those floats should
match (most of the time) the ground-truth label.

In [ ]:
# Step-by-step inference on one image. Order of DMA operations matches
# the ICTP gamma/neutron reference (which is known to work on this board):
#   1. write to IP CTRL to set ap_start
#   2. push send channel
#   3. arm recv channel
#   4. wait for send
#   5. wait for recv
import time

idx = 0
img = pixels[idx].reshape(32, 32, 3)
label = labels[idx]

print('1. Packing image into input buffer...')
t0 = time.time()
image_to_axis_buffer(pixels[idx], in_buffer)
print(f'   done in {(time.time()-t0)*1000:.1f} ms')

ctrl = ip.read(0x00)
print(f'2. Pre-check IP: CTRL=0x{ctrl:08x} (ap_idle={(ctrl>>2)&1})')

print('3. Triggering IP (write 1 to CTRL)...')
ip.write(0x00, 1)

print('4. Push send channel...')
dma_send.transfer(in_buffer)

print('5. Arm recv channel...')
dma_recv.transfer(out_buffer)

print('6. Wait for send...')
t0 = time.time()
dma_send.wait()
print(f'   send done in {(time.time()-t0)*1000:.2f} ms')

print('7. Wait for recv...')
t0 = time.time()
dma_recv.wait()
print(f'   recv done in {(time.time()-t0)*1000:.2f} ms')

ctrl = ip.read(0x00)
print(f'8. Post-execution IP: CTRL=0x{ctrl:08x} (ap_done={(ctrl>>1)&1}, ap_idle={(ctrl>>2)&1})')

scores = axis_buffer_to_floats(out_buffer)
pred = int(np.argmax(scores))
print('\nRaw scores:')
for i, s in enumerate(scores):
    marker = ' <-- argmax' if i == pred else ''
    print(f'  class {i} ({CIFAR10_CLASSES[i]:>10s}): {s:+.5f}{marker}')
print()
print(f'Predicted: {pred} ({CIFAR10_CLASSES[pred]})')
print(f'Truth:     {label} ({CIFAR10_CLASSES[label]})')
print('CORRECT' if pred == label else 'WRONG')

## 9. Inference helper

Now wrap the steps above into a single function so the next cells can iterate without repeating boilerplate.

In [ ]:
def inference_image(img):
    """
    Run one inference pass on the hardware accelerator.

    Uses the same order of DMA operations as the ICTP gamma/neutron
    reference notebook on this board:
        IP start -> send -> recv -> send.wait -> recv.wait
    """
    image_to_axis_buffer(img, in_buffer)
    ip.write(0x00, 1)
    dma_send.transfer(in_buffer)
    dma_recv.transfer(out_buffer)
    dma_send.wait()
    dma_recv.wait()
    return axis_buffer_to_floats(out_buffer)

## 10. Smoke test: 10 images

Before running over the full 10 000-sample test set (which may take a
few minutes given the per-pixel Python loop in `image_to_axis_buffer`),
confirm with 10 images that the pipeline is stable and that accuracy is
in the right ballpark (around 60%).

In [ ]:
n_smoke = 10
correct = 0
for i in range(n_smoke):
    scores = inference_image(pixels[i])
    pred = int(np.argmax(scores))
    ok = pred == labels[i]
    correct += int(ok)
    print(f'  i={i:2d}  pred={pred} ({CIFAR10_CLASSES[pred]:>10s})  true={labels[i]} ({CIFAR10_CLASSES[labels[i]]:>10s})  {"ok" if ok else "WRONG"}')
print(f'\nSmoke-test accuracy: {correct}/{n_smoke}')

## 11. Full test set

Run inference over every test sample, measuring wall-clock latency.
Expected ballpark: 60-61% accuracy (the bit-accurate hls4ml simulation
on the host gave 60.97%, the software QKeras model gave 61.41%).

In [ ]:
correct = 0
preds = []

t0 = time.time()
for i, (img, label) in enumerate(zip(pixels, labels)):
    scores = inference_image(img)
    pred = int(np.argmax(scores))
    preds.append(pred)
    if pred == label:
        correct += 1
    if i % 1000 == 0:
        elapsed = time.time() - t0
        print(f'  {i:5d}/{N}  elapsed={elapsed:6.1f}s  running acc={correct/max(i,1):.4f}')

t1 = time.time()
total = t1 - t0

accuracy = correct / N
throughput = N / total
latency_ms = (total / N) * 1000

print()
print(f'Hardware accuracy on {N} samples: {accuracy:.4f}')
print(f'Total wall-clock time: {total:.1f} s')
print(f'Per-image average: {latency_ms:.2f} ms (includes Python overhead)')
print(f'Throughput: {throughput:.1f} images/s')

## 12. Confusion matrix

Per-class behaviour. With ~14 K parameters the model is small, so
expect confusion between visually similar classes (cat/dog, deer/horse,
airplane/ship, automobile/truck).

In [ ]:
cm = confusion_matrix_np(labels, np.array(preds), 10)
plot_confusion_matrix(cm, class_names=CIFAR10_CLASSES, normalize=False)

## 13. Pure-hardware latency (no Python overhead)

The figure in step 11 is dominated by the per-pixel Python loop in
`image_to_axis_buffer` (3072 calls to `float_to_axis_word` per image,
all interpreted). To get a fair estimate of the **hardware** latency,
pre-pack the input buffer once and only measure the DMA + IP execution
in the loop.

In [ ]:
# Pre-pack one image into the input buffer.
image_to_axis_buffer(pixels[0], in_buffer)

# Warm up
for _ in range(20):
    ip.write(IP_CTRL_REG, 1)
    dma_recv.transfer(out_buffer)
    dma_send.transfer(in_buffer)
    dma_send.wait()
    dma_recv.wait()

n_meas = 1000
t0 = time.time()
for _ in range(n_meas):
    ip.write(IP_CTRL_REG, 1)
    dma_recv.transfer(out_buffer)
    dma_send.transfer(in_buffer)
    dma_send.wait()
    dma_recv.wait()
t1 = time.time()

hw_total = t1 - t0
hw_per_image_us = (hw_total / n_meas) * 1e6
hw_throughput = n_meas / hw_total
print(f'Hardware-only loop ({n_meas} iterations on the same pre-packed image):')
print(f'  total = {hw_total:.3f} s')
print(f'  per-image = {hw_per_image_us:.1f} us')
print(f'  throughput = {hw_throughput:,.0f} images/s')

----

### Appendix: recipe to regenerate `test_dataset/cifar10_test.csv`

Run this **on the host** (not on the board) inside the `neuralEnv10`
environment to write a CSV of normalised CIFAR-10 test images that the
notebook can ingest:

```python
import os
import numpy as np
import pandas as pd
from tensorflow.keras.datasets import cifar10

(_, _), (x_test, y_test) = cifar10.load_data()
x = (x_test.astype(np.float32) / 255.0).reshape(len(x_test), -1)
y = y_test.flatten().astype(int)[:, None]
out = np.hstack([x, y])
cols = [f'p{i}' for i in range(x.shape[1])] + ['label']
os.makedirs('test_dataset', exist_ok=True)
pd.DataFrame(out, columns=cols).to_csv('test_dataset/cifar10_test.csv', index=False)
```

Copy the resulting `test_dataset/cifar10_test.csv` next to this notebook
on the board. Approximate size: ~370 MB for the full 10 000 test
samples; you may want to subset to e.g. 1000 samples for faster
iteration.